# TIP PREDICTION MODEL ANALYSIS (FHVHV)

In [ ]:
from pyspark.sql import SparkSession, DataFrame, Window
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.functions import vector_to_array
from pyspark.sql import functions as f
import findspark
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
from sklearn.metrics import confusion_matrix
from pathlib import Path
import os
import numpy as np

from yellowTipModel import train_test_split

In [ ]:
projectRoot = Path.cwd().parents[2]

DATA_PATH = Path(projectRoot) / "data" / "clean" / "transformed_tlc_trip_record" / "fhvhv"
BINARY_MODEL_PATH = Path(projectRoot) / "src" / "model" / "tipPrediction" / "trainedModels" / "fhvhv_model" / "binary"
MULTICLASS_MODEL_PATH = Path(projectRoot) / "src" / "model" / "tipPrediction" / "trainedModels" / "fhvhv_model" / "multiclass"
TABLES_DIR = projectRoot / "data" / "plotTables" / "tipModelAnalysis" / "fhvhv"
os.makedirs(TABLES_DIR, exist_ok=True)

TEST_CUT = "2025-01-01"

NUMERIC_FEATURES = [
    "temperature_2m",
    "apparent_temperature",
    "precipitation",
    "rain",
    "snowfall",
    "wind_speed_10m",
    "wind_gusts_10m",
    "relative_humidity_2m",
    "weather_code",
    "trip_miles",
    "total_amount",
    "pu_poi_education_per_mi2",
    "pu_poi_entertainment_per_mi2",
    "pu_poi_food_drink_per_mi2",
    "pu_poi_healthcare_per_mi2",
    "pu_poi_nightlife_alcohol_per_mi2",
    "pu_poi_tourism_per_mi2",
    "do_poi_education_per_mi2",
    "do_poi_entertainment_per_mi2",
    "do_poi_food_drink_per_mi2",
    "do_poi_healthcare_per_mi2",
    "do_poi_nightlife_alcohol_per_mi2",
    "do_poi_tourism_per_mi2",
]

CATEGORICAL_FEATURES = [
    "dropoff_borough",
    'pickup_borough',
    "PULocationID",
    "pickup_day_of_week",
    "time_of_day",
    "DOLocationID"
]
FEATURES_COL = NUMERIC_FEATURES + CATEGORICAL_FEATURES

In [ ]:
def save_table(df:pd.DataFrame, name:str, path: Path):
    path_table = os.path.join(str(path), f"{name}.csv")
    df.to_csv(path_table, index=False)

def load_csv(path:Path) -> pd.DataFrame:
    return pd.read_csv(path)

# Load Data

In [ ]:
findspark.init()

spark = (SparkSession.builder
        .master("local[6]")  
        .config("spark.driver.memory", "8g")
        .config("spark.executor.memory", "8g")
        .config("spark.sql.shuffle.partitions", "100")
        .config("spark.sql.adaptive.advisoryPartitionSizeInBytes", "64m")
        .config("spark.driver.maxResultSize", "1g")
        .config("spark.sql.session.timeZone", "America/New_York")
        .config("spark.sql.timestampType", "TIMESTAMP_NTZ")
        # Network binding fixes for macOS
        .config("spark.driver.host", "127.0.0.1")
        .config("spark.driver.bindAddress", "127.0.0.1")
        .getOrCreate())

In [ ]:
df = spark.read.parquet(str(DATA_PATH))

In [ ]:
train, test = train_test_split(df, TEST_CUT)
print(f"Train: {train.count():,}, Test: {test.count():,}")

In [ ]:
train_binary = train.withColumn("has_tip", (f.col("tip_category") > 0).cast("int"))
test_binary = test.withColumn("has_tip", (f.col("tip_category") > 0).cast("int"))

# Load Model Pipeline

In [ ]:
binary_pipeline = PipelineModel.load(str(BINARY_MODEL_PATH))
multiclass_pipeline = PipelineModel.load(str(MULTICLASS_MODEL_PATH))

# Prediction

In [ ]:
preds_binary = binary_pipeline.transform(test_binary)

test_with_tip = preds_binary.filter(f.col("prediction") == 1)

test_multiclass = (
    test_with_tip
    .filter(f.col("tip_category") > 0)
    .withColumn(
        "tip_category_indexed", 
        (f.col("tip_category") - 1).cast("int")
    )
)

preds_multiclass = multiclass_pipeline.transform(test_multiclass)

In [ ]:
preds = preds_multiclass.withColumn(
    "prob_array", 
    vector_to_array(f.col("probability_multi"))
)

for i in range(4):
    preds = preds.withColumn(f"prob_class_{i + 1}", f.col("prob_array")[i])

preds = preds.withColumnRenamed("prob_class_1", "prob_Rango1_$1-2")
preds = preds.withColumnRenamed("prob_class_2", "prob_Rango2_$2-3")
preds = preds.withColumnRenamed("prob_class_3", "prob_Rango3_$3-5")
preds = preds.withColumnRenamed("prob_class_4", "prob_Rango4_$5+")

# Plots

In [ ]:
preds.show()

## Confusion Matrix

In [ ]:
def plot_confusion_matrix_plotly(preds):

    y_true = preds.select("tip_category").toPandas().values.flatten()
    y_pred = preds.select("prediction").toPandas().values.flatten()
    
    cm = confusion_matrix(y_true, y_pred)
    labels = ['$0', '$1-2', '$2-3', '$3-5', '$5+']

    fig = go.Figure(data=go.Heatmap(
        z=cm,
        x=labels,
        y=labels,
        text=cm,
        texttemplate='%{text}',
        textfont={"size": 16},
        colorscale='Blues',
        hoverongaps=False,
        hovertemplate='Real: %{y}<br>Predicted: %{x}<br>Count: %{z}<extra></extra>'
    ))
    
    fig.update_layout(
        title='Confusion Matrix',
        xaxis_title='Prediction',
        yaxis_title='Real',
        width=600,
        height=500,
        font=dict(size=12)
    )
    
    fig.show()

def plot_confusion_matrix_binary(preds):
    
    y_true = preds.select("has_tip").toPandas().values.flatten()
    y_pred = preds.select("prediction").toPandas().values.flatten()
    
    
    cm = confusion_matrix(y_true, y_pred)

    labels = ['Tips', 'No tips']
    
    fig = go.Figure(data=go.Heatmap(
        z=cm,
        x=labels,
        y=labels,
        text=cm,
        texttemplate='%{text}',
        textfont={"size": 16},
        colorscale='Blues',
        hoverongaps=False,
        hovertemplate='Real: %{y}<br>Predicted: %{x}<br>Count: %{z}<extra></extra>'
    ))
    
    fig.update_layout(
        title='Confusion Matrix - Tips Prediction (binary)',
        xaxis_title='Predited',
        yaxis_title='Real',
        width=600,
        height=500,
        font=dict(size=12)
    )
    
    fig.show()

In [ ]:
plot_confusion_matrix_binary(preds_binary)

In [ ]:
plot_confusion_matrix_plotly(preds)

# Probability Distribution

In [ ]:
def plot_probability_distribution_plotly(preds):
    """
    Probability distribution for each class
    """
    # Extraer probabilidades
    pdf = preds.select(
        "tip_category",
        f.col("prob_Rango1_$1-2").alias("prob_$1-2"),
        f.col("prob_Rango2_$2-3").alias("prob_$2-3"),
        f.col("prob_Rango3_$3-5").alias("prob_$3-5"),
        f.col("prob_Rango4_$5+").alias("prob_$5+")
    ).toPandas()

    fig = go.Figure()
    colores = ['blue', 'green', 'orange', 'purple']
    clases = ['$1-2', '$2-3', '$3-5', '$5+']
    
    for i, clase in enumerate(clases):
        fig.add_trace(go.Histogram(
            x=pdf[f'prob_{clase}'],
            name=f'Prob {clase}',
            opacity=0.6,
            marker_color=colores[i],
            nbinsx=30
        ))
    
    fig.update_layout(
        title='Histogram of Prediction Probabilities',
        xaxis_title='Probability',
        yaxis_title='Frequency',
        barmode='overlay',
        height=500,
        width=800
    )
    
    fig.show()

In [ ]:
plot_probability_distribution_plotly(preds)

## Temporal Accuracy

In [ ]:
def plot_temporal_accuracy_plotly(preds):
    """
    Accuracy by hour of the day
    """

    correct = (
        preds
        .withColumn("correct", f.col("tip_category") == f.col("prediction"))
    )

    acc_by_hour = (
        correct
        .groupBy("pickup_hour")
        .agg(
            f.mean(f.col("correct").cast("double")).alias("accuracy"),
            f.count("*").alias("count")
        )
        .orderBy("pickup_hour")
    ).toPandas()
    
    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=acc_by_hour['pickup_hour'],
            y=acc_by_hour['accuracy'],
            mode='lines+markers',
            name='Precisión',
            line=dict(color='blue', width=2),
            marker=dict(size=8),
            text=acc_by_hour['count'],
            hovertemplate='Hora: %{x}<br>Precisión: %{y:.2%}<br>Muestras: %{text:,}<extra></extra>'
        ),
    )
    fig.update_layout(
        height=700,
        showlegend=False,
        title_text="Análisis Temporal de Precisión del Modelo"
    )
    
    fig.show()

In [ ]:
plot_temporal_accuracy_plotly(preds)

In [ ]:
spark.stop()